In [1]:
%pip install -qU google-generativeai
%pip install -qU google-ai-generativelanguage==0.6.15
%pip install -qU langchain-google-genai
%pip install -qU langchain-community
%pip install -qU langgraph
%pip install -qU python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import re
import google.generativeai as genai
from typing import TypedDict
from langgraph.graph import StateGraph, END

c:\Desarrollo\MULTIAGENTES_LANGGRAPH\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Usuario\AppData\Local\Temp\ipykernel_24056\2348446817.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

# Cargar las variables desde el archivo .env
load_dotenv()

# Obtener las API Keys del entorno
GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

# Configurar el SDK de Google con tu llave
genai.configure(api_key=GOOGLE_API_KEY)

# Inicializar el modelo y probarlo
model = genai.GenerativeModel('gemini-2.5-flash')
response = model.generate_content("Hola mundo")
print(response.text)

¡Hola! ¿Cómo estás?


In [6]:
import google.generativeai as genai

class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        # Para Gemini, el rol del sistema se configura directo en las instrucciones
        self.model_name = "gemini-2.5-flash"

    def __call__(self, message):
        self.messages.append({"role": "user", "parts": [message]})
        result = self.execute()
        self.messages.append({"role": "model", "parts": [result]})
        return result

    def execute(self):
        # Configuramos el modelo inyectando las instrucciones del sistema si existen
        if self.system:
            client = genai.GenerativeModel(
                model_name=self.model_name,
                system_instruction=self.system
            )
        else:
            client = genai.GenerativeModel(model_name=self.model_name)
            
        # Le pasamos TODO el historial acumulado en self.messages
        completion = client.generate_content(self.messages)
        return completion.text

In [ ]:
agente = Agent(system="Eres un asistente útil y objetivo.")
print(agente("Cuál es el equipo de futbol favorito del Mundial 2026, entre Argentina y Argelia?"))

Entre Argentina y Argelia, **Argentina es, sin ninguna duda, el equipo de fútbol favorito y con mucha más probabilidad de ser candidato en el Mundial 2026.**

Aquí las razones:

1.  **Argentina:**
    *   **Campeón del Mundo vigente:** Ganaron la Copa Mundial de la FIFA 2022, lo que los posiciona inmediatamente como uno de los principales contendientes para el próximo torneo.
    *   **Historial:** Son una potencia histórica del fútbol mundial, con tres títulos mundiales (1978, 1986, 2022) y numerosas apariciones en finales.
    *   **Plantilla:** Cuentan con una base de jugadores de clase mundial, muchos de ellos en su mejor momento o entrando en él, incluso si Lionel Messi no participa en 2026, el equipo tiene una estructura sólida y talento en todas las líneas.
    *   **Experiencia:** Tienen un cuerpo técnico y jugadores con experiencia en ganar torneos importantes.

2.  **Argelia:**
    *   **Potencia Regional:** Argelia es una fuerza importante en el fútbol africano, habiendo gan

In [7]:
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas - y luego retorna "PAUSA".
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:

consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej: "consultar_stock: teclado")

consultar_precio_producto: devuelve el precio unitario de un producto (ej: "consultar_precio_producto: mouse gamer")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en el inventario?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en el inventario.
Respuesta: Hay 75 monitores en el inventario.
""".strip()

In [8]:
def consultar_stock(item: str) -> str:
    """
    Simula la consulta de stock de item en el inventario.
    """
    item = item.lower().strip()
    stock = {
        "monitor": 75,
        "teclado": 120,
        "mouse de gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impresora": 15
    }

    if item in stock:
        return f"Tenemos {stock[item]} {item}s en stock."
    else:
        return f"Item '{item}' no encontrado en el inventario."


def consultar_precio_producto(producto: str) -> str:
    """
    Simula la consulta del precio unitario de un producto.
    """
    producto = producto.lower().strip()
    precios = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }

    if producto in precios:
        return f"El precio de un(a) {producto} es USD {precios[producto]:.2f}."
    else:
        return f"Producto '{producto}' no hallado en la lista de precios."

In [ ]:
print(consultar_stock("teclado"))
print(consultar_precio_producto("impresora"))
print(consultar_stock("monitor"))
print(consultar_stock("sillas"))

Tenemos 120 teclados en stock.
El precio de un(a) impresora es USD 750.00.
Tenemos 75 monitors en stock.
Item 'sillas' no encontrado en el inventario.


In [9]:
import re
import google.generativeai as genai

def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')
    # Iniciamos el chat vacío
    chat = model.start_chat(history=[])

    # Le inyectamos las reglas del sistema (PROMPT_REACT) como primer mensaje
    chat.send_message(PROMPT_REACT)

    # El primer mensaje real que procesa el ciclo es la pregunta del usuario
    proximo_mensaje = f"Pregunta: {pregunta}"

    for i in range(max_iterations):
        # Enviamos el mensaje al chat (mantiene el historial automáticamente)
        response = chat.send_message(proximo_mensaje)
        response_text = response.text.strip()

        print(f"--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")

        # 1. Si el modelo ya nos da la respuesta final, terminamos con éxito
        if "Respuesta:" in response_text:
            # Extraemos lo que está después de "Respuesta:"
            partes = response_text.split("Respuesta:")
            return partes[-1].strip()

        # 2. Si no es respuesta, buscamos qué acción quiere ejecutar
        match = re.search(r"Acción:\s*(\w+):\s*(.*)", response_text)

        if match:
            action_name = match.group(1).strip()
            # Limpiamos el argumento por si viene con un '\nPAUSA' pegado abajo
            action_arg = match.group(2).split("\n")[0].strip()

            # Ejecutamos la función de Python correspondiente
            if action_name == "consultar_stock":
                observacion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion = consultar_precio_producto(action_arg)
            else:
                observacion = f"Error: Acción '{action_name}' desconocida."

            print(f"-> [Ejecutando herramienta]: {action_name}({action_arg})")
            print(f"-> [Observación obtenida]: {observacion}\n")

            # El próximo mensaje que le mandamos al chat es la observación pura
            proximo_mensaje = f"Observación: {observacion}"

        else:
            # Si el modelo no respondió con el formato ReAct correcto
            return f"Error: El agente se desvió del formato ReAct en la iteración {i+1}. Respuesta: {response_text}"

    return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [ ]:
pregunta_1 = "¿Cuántos mouse de gamer están disponibles en el inventario?"
print(f"**Interacción 1: {pregunta_1}**\n")

respuesta_1 = run_react_agent(pregunta_1)

print(f"\n**RESPUESTA FINAL DEL AGENTE 1:** {respuesta_1}\n")
print("\n" + "="*50 + "\n")

**Interacción 1: ¿Cuántos mouse de gamer están disponibles en el inventario?**

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de "mouse de gamer".
Acción: consultar_stock: mouse de gamer
PAUSA

-> [Ejecutando herramienta]: consultar_stock(mouse de gamer)
-> [Observación obtenida]: Tenemos 80 mouse de gamers en stock.

--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: Hay 80 mouse de gamers en stock.


**RESPUESTA FINAL DEL AGENTE 1:** Hay 80 mouse de gamers en stock.





In [ ]:
pregunta_2 = "¿Cuál es el precio de una impresora?"
print(f"**Interacción 2: {pregunta_2}**\n")

respuesta_2 = run_react_agent(pregunta_2)

print(f"\n**RESPUESTA FINAL DEL AGENTE 2:** {respuesta_2}\n")
print("\n" + "="*50 + "\n")

**Interacción 2: ¿Cuál es el precio de una impresora?**

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_precio_producto para saber el precio de una impresora.
Acción: consultar_precio_producto: impresora
PAUSA

-> [Ejecutando herramienta]: consultar_precio_producto(impresora)
-> [Observación obtenida]: El precio de un(a) impresora es USD 750.00.

--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: El precio de una impresora es USD 750.00.


**RESPUESTA FINAL DEL AGENTE 2:** El precio de una impresora es USD 750.00.





In [ ]:
pregunta_3 = "¿Tenemos sillas en inventario?"
print(f"**Interacción 3: {pregunta_3}**\n")

respuesta_3 = run_react_agent(pregunta_3)

print(f"\n**RESPUESTA FINAL DEL AGENTE 3:** {respuesta_3}\n")

**Interacción 3: ¿Tenemos sillas en inventario?**

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: El usuario quiere saber si tenemos sillas en inventario. Para responder a esto, necesito consultar la cantidad de sillas disponibles utilizando la acción `consultar_stock`.
Acción: consultar_stock: silla
PAUSA

-> [Ejecutando herramienta]: consultar_stock(silla)
-> [Observación obtenida]: Item 'silla' no encontrado en el inventario.

--- Iteración 2 ---
Modelo pensó/respondió:
Pensamiento: La observación indica que "silla" no fue encontrado en el inventario. Esto significa que no hay sillas disponibles.
Respuesta: No, no tenemos sillas en el inventario.


**RESPUESTA FINAL DEL AGENTE 3:** No, no tenemos sillas en el inventario.



In [ ]:
pregunta_4 = "¿Cuál es el producto más costoso?"
print(f"**Interacción 4: {pregunta_4}**\n")

respuesta_4 = run_react_agent(pregunta_4)

print(f"\n**RESPUESTA FINAL DEL AGENTE 4:** {respuesta_4}\n")

**Interacción 4: ¿Cuál es el producto más costoso?**

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Para determinar el producto más costoso, necesitaría una lista de todos los productos disponibles en el inventario y poder consultar el precio de cada uno. Las herramientas disponibles (`consultar_stock`, `consultar_precio_producto`) solo me permiten consultar información sobre un producto específico que yo ya conozca. No puedo listar todos los productos ni compararlos para encontrar el más caro con las herramientas actuales. No puedo responder a esta pregunta directamente.
Respuesta: Lo siento, no puedo determinar cuál es el producto más costoso con las herramientas que tengo. Necesitaría una lista de todos los productos y poder consultar sus precios para hacer esa comparación.


**RESPUESTA FINAL DEL AGENTE 4:** Lo siento, no puedo determinar cuál es el producto más costoso con las herramientas que tengo. Necesitaría una lista de todos los productos y poder consultar sus pre

In [10]:
def herramienta_encontrar_producto_mas_costoso() -> str:
    """
    Retorna el nombre y el precio del producto más costoso en el inventario.
    Esta función no requiere argumentos adicionales.
    """
    precios_del_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00  # Corregido a una sola 's' para mantener consistencia
    }

    if not precios_del_inventario: 
        return "Lo sentimos, no hallamos ningún producto en la lista de precios para su comparación."

    # Encontrar el producto con el precio máximo
    nombre_producto_mas_costoso = max(precios_del_inventario, key=precios_del_inventario.get)
    valor_producto_mas_costoso = precios_del_inventario[nombre_producto_mas_costoso]

    return f"El producto más costoso es el(la) {nombre_producto_mas_costoso} con precio de USD {valor_producto_mas_costoso:.2f}."

In [11]:
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.

Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas, y luego regresa "PAUSA".
La "Observación" será el resultado de la acción ejecutada.

Acciones disponibles:
- consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej.: "consultar_stock: teclado")
- consultar_precio_producto: devuelve el precio unitario de un producto (ej.: "consultar_precio_producto: mouse gamer")
- encontrar_producto_mas_costoso: devuelve el nombre y el precio del producto más costoso del inventario (no requiere argumentos, ej.: "encontrar_producto_mas_costoso")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en stock?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en stock.
Respuesta: Hay 75 monitores en stock.

Ejemplo:
Pregunta: ¿Cuál es el producto más costoso?
Pensamiento: Necesito usar la acción encontrar_producto_mas_costoso para descubrir qué producto tiene el precio más alto.
Acción: encontrar_producto_mas_costoso
PAUSA

Observación: El producto más costoso es el monitor con un precio de USD 999.90.
Respuesta: El producto más costoso es el monitor con un precio de USD 999.90.
""".strip()

In [12]:
import re
import google.generativeai as genai

def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    """
    Ejecuta el ciclo ReAct para una determinada pregunta usando el modelo Gemini.
    """
    model = genai.GenerativeModel('gemini-2.5-flash')
    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pregunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")

        # Verificar si es la respuesta final
        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        # Parsear la acción y su argumento opcional
        match = re.search(r"Acción:\s*(\w+)(?::\s*([^\n]*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacion_de_accion = ""

            # Mapeo dinámico de herramientas
            if action_name == "consultar_stock":
                observacion_de_accion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion_de_accion = consultar_precio_producto(action_arg)
            elif action_name == "encontrar_producto_mas_costoso": 
                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
            else:
                observacion_de_accion = f"Error: Acción '{action_name}' desconocida. Verifica el prompt o la implementación de la herramienta."

            # Alimentamos la observación para la siguiente vuelta del bucle
            current_prompt = f"Observación: {observacion_de_accion}"

            print(f"Ejecutó acción: {action_name} con argumento '{action_arg}'")
            print(f"Observación: {observacion_de_accion}\n")

        else:
            return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La Última respuesta fue: {response_text}"

    return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

In [ ]:
pregunta_4 = "¿Cuál es el producto más costoso?"
print(f"**Interacción 4: {pregunta_4}**\n")

respuesta_4 = run_react_agent(pregunta_4)

print(f"\n**RESPUESTA FINAL DEL AGENTE 4:** {respuesta_4}\n")

**Interacción 4: ¿Cuál es el producto más costoso?**


--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Necesito usar la acción encontrar_producto_mas_costoso para descubrir qué producto tiene el precio más alto.
Acción: encontrar_producto_mas_costoso
PAUSA

Ejecutó acción: encontrar_producto_mas_costoso con argumento ''
Observación: El producto más costoso es el(la) monitor con precio de USD 999.90.


--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: El producto más costoso es el monitor con un precio de USD 999.90.


**RESPUESTA FINAL DEL AGENTE 4:** El producto más costoso es el monitor con un precio de USD 999.90.



In [13]:
import re
import google.generativeai as genai

def run_react_agent_with_history(pregunta: str, max_iterations: int = 5) -> tuple[str, list]: 
    """
    Ejecuta el ciclo ReAct manteniendo y retornando el historial completo de la conversación.
    """
    model = genai.GenerativeModel('gemini-2.5-flash') 
    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT) 

    current_prompt = pregunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")

        # Verificar si es la respuesta final (Retorna texto e historial)
        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip(), chat.history

        # Parsear la acción
        match = re.search(r"Acción:\s*(\w+)(?::\s*([^\n]*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacion_de_accion = ""

            if action_name == "consultar_stock":
                observacion_de_accion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion_de_accion = consultar_precio_producto(action_arg)
            elif action_name == "encontrar_producto_mas_costoso":
                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
            else:
                observacion_de_accion = f"Error: Acción '{action_name}' desconocida. Verifica el prompt o la implementación de la herramienta."

            current_prompt = f"Observación: {observacion_de_accion}"

            print(f"Ejecutó la acción: {action_name} con argumento '{action_arg}'")
            print(f"Observación: {observacion_de_accion}\n")

        else:
            # Corregido para que devuelva correctamente la tupla (Error, Historial)
            error_msg = f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La Última respuesta fue: {response_text}"
            return error_msg, chat.history

    return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final.", chat.history

In [14]:
pregunta_ejemplo = "¿Cuántos teclados tenemos en stock?"
# Ejecutamos el agente que devuelve la tupla (respuesta, historial)
respuesta_ejemplo, historial_completo = run_react_agent_with_history(pregunta_ejemplo)

print(f"**RESPUESTA FINAL DEL AGENTE:** {respuesta_ejemplo}\n")

print("\n--- Historial Completo de la Interacción ---")
for i, message in enumerate(historial_completo):
    # message.role nos dice si habló el 'user' o el 'model'
    print(f"--- Mensaje {i+1} (rol: {message.role}) ---")

    for part in message.parts:
        if hasattr(part, 'text'):
            print(part.text)
        else:
            print(part)
    print("-" * 20)

print("\n--- Fin del historial ---\n")


--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de teclados.
Acción: consultar_stock: teclado
PAUSA

Ejecutó la acción: consultar_stock con argumento 'teclado'
Observación: Tenemos 120 teclados en stock.


--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: Hay 120 teclados en stock.

**RESPUESTA FINAL DEL AGENTE:** Hay 120 teclados en stock.


--- Historial Completo de la Interacción ---
--- Mensaje 1 (rol: user) ---
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.

Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas, y luego regresa "PAUSA".
La "Observación" será el resultado de la acción ejecutada.

Acciones disponibles:
- consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej.: "consultar_stock: teclado")
- consultar_precio_producto: devuelve el precio unitario de un product

In [15]:
def herramienta_calcular_valor_total_lista(lista_items: str) -> str:
    """
    Calcula el valor total de una lista de items de compra.
    Recibe una string con items separados por coma (ej: "teclado, mouse de gamer, monitor").
    """
    precios_del_inventario = { 
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }

    # Separamos por coma y limpiamos espacios/mayúsculas
    items_procesados = [item.strip().lower() for item in lista_items.split(',')]

    valor_total = 0.0
    items_no_encontrados = []

    for item in items_procesados: 
        if item in precios_del_inventario:
            valor_total += precios_del_inventario[item]
        else:
            items_no_encontrados.append(item)

    respuesta = f"El valor total de los items encontrados es USD {valor_total:.2f}."
    if items_no_encontrados:
        # Corregido: "incluidos" va sin tilde
        respuesta += f" Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: {', '.join(items_no_encontrados)}."

    return respuesta

In [16]:
print("--- Testando herramienta_calcular_valor_total_lista --- \n")

# Test 1: Todos los ítems existen
lista_1 = "teclado, mouse de gamer, monitor"
resultado_1 = herramienta_calcular_valor_total_lista(lista_1)
print(f"Lista 1: '{lista_1}'")
print(f"Resultado 1: {resultado_1}\n")

print("-" * 30 + "\n")

# Test 2: Incluye un ítem que NO existe (ej. "gabinete") y espacios raros
lista_2 = "  teclado, gabinete, WEBCAM "
resultado_2 = herramienta_calcular_valor_total_lista(lista_2)
print(f"Lista 2: '{lista_2}'")
print(f"Resultado 2: {resultado_2}\n")

--- Testando herramienta_calcular_valor_total_lista --- 

Lista 1: 'teclado, mouse de gamer, monitor'
Resultado 1: El valor total de los items encontrados es USD 1249.40.

------------------------------

Lista 2: '  teclado, gabinete, WEBCAM '
Resultado 2: El valor total de los items encontrados es USD 270.00. Los siguientes items no fueron encontrados y tampoco incluidos en el cálculo: gabinete.



In [29]:
#                ***   La forma del profe  ***
#def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
#    model = genai.GenerativeModel('gemini-2.5-flash')
#
#    chat = model.start_chat(history=[])
#    chat.send_message(PROMPT_REACT)

#    current_prompt = pregunta

#    for i in range(max_iterations):
#        response = chat.send_message(current_prompt)
#        response_text = response.text.strip()

#        print(f"\n--- Iteración {i+1} ---")
#        print(f"Modelo pensó/respondió:\n{response_text}\n")

        # 1. Verificar si es la respuesta final
#        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
#        if response_match_final:
#            return response_match_final.group(1).strip()

        # 2. Si no es respuesta final, buscar la Acción
#        match = re.search(r"Acción:\s*(\w+)(?::\s*([^\n]*))?", response_text)

#        if match:
#            action_name = match.group(1).strip()
#            action_arg = match.group(2).strip() if match.group(2) is not None else ""

#            observacion_de_accion = ""

#            if action_name == "consultar_stock":
#                observacion_de_accion = consultar_stock(action_arg)
#            elif action_name == "consultar_precio_producto":
#                observacion_de_accion = consultar_precio_producto(action_arg)
            # Corregido de portugués a español: "producto_mas_costoso"
#            elif action_name == "encontrar_producto_mas_costoso":
#                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
#            elif action_name == "calcular_valor_total_lista":
#                observacion_de_accion = herramienta_calcular_valor_total_lista(action_arg)
#            else:
#                observacion_de_accion = f"Error: Acción '{action_name}' desconocida. Verifica el prompt o la implementación de la herramienta."

#            current_prompt = f"Observación: {observacion_de_accion}"

#            print(f"Ejecutó acción: {action_name} con argumento '{action_arg}'")
#            print(f"Observación: {observacion_de_accion}\n")

#        else:
#            return f"Error: El agente no logró extraer una Acción o Respuesta final tras {i+1} iteraciones. La Última respuesta fue: {response_text}"

#    return "Error: Límite máximo de iteraciones alcanzado sin una respuesta final."

#                ### Refatorizado por tiempo ###
#import re
#import time
#import google.api_core.exceptions  # <-- Importante para atrapar el error de cuota

#def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
#    model = genai.GenerativeModel('gemini-2.5-flash')

#    chat = model.start_chat(history=[])
    
    # Manejo de cuota para el mensaje inicial
#    try:
#        chat.send_message(PROMPT_REACT)
#    except google.api_core.exceptions.ResourceExhausted:
#        print("⚠️ API saturada al iniciar. Esperando 30 segundos de enfriamiento...")
#        time.sleep(30)
#        chat.send_message(PROMPT_REACT)

#    current_prompt = pregunta

#    for i in range(max_iterations):
#        response_text = ""
        
        # Bucle de reintento inteligente si la API nos frena el paso
#        for intento in range(3):
#            try:
#                response = chat.send_message(current_prompt)
#                response_text = response.text.strip()
#                break # Si funcionó, salimos del bucle de reintentos
#            except google.api_core.exceptions.ResourceExhausted as e:
#                print(f"🛑 Error de Cuota detectado en Iteración {i+1} (Intento {intento+1}/3).")
#                print("Esperando 45 segundos para que Google libere la IP...")
#                time.sleep(45) # Le damos el tiempo necesario para limpiar el contador
#                if intento == 2: # Si falló 3 veces seguidas, tiramos el error para no colgar el script
#                    raise e

#        print(f"\n--- Iteración {i+1} ---")
#        print(f"Modelo pensó/respondió:\n{response_text}\n")

        # 1. Verificar si es la respuesta final
#        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
#        if response_match_final:
#            return response_match_final.group(1).strip()

        # 2. Si no es respuesta final, buscar la Acción
#        match = re.search(r"Acción:\s*(\w+)(?::\s*([^\n]*))?", response_text)

#        if match:
#            action_name = match.group(1).strip()
#            action_arg = match.group(2).strip() if match.group(2) is not None else ""

#            observacion_de_accion = ""

#            if action_name == "consultar_stock":
#                observacion_de_accion = consultar_stock(action_arg)
#            elif action_name == "consultar_precio_producto":
#                observacion_de_accion = consultar_precio_producto(action_arg)
#            elif action_name == "encontrar_producto_mas_costoso":
#                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
#            elif action_name == "calcular_valor_total_lista":
#                observacion_de_accion = herramienta_calcular_valor_total_lista(action_arg)
#            else:
#                observacion_de_accion = f"Error: Acción '{action_name}' desconocida."

#            current_prompt = f"Observación: {observacion_de_accion}"

#            print(f"Ejecutó acción: {action_name} con argumento '{action_arg}'")
#            print(f"Observación: {observacion_de_accion}\n")

#        else:
#            return f"Error: No se extrajo Acción ni Respuesta. Última respuesta: {response_text}"

#    return "Error: Límite máximo de iteraciones alcanzado."

#                          USANDO GROQ
from dotenv import load_dotenv
import os

load_dotenv() # Esto lee el archivo .env y carga las variables

import re
from groq import Groq

# Inicializa el cliente. 
# Si guardaste la key en el archivo .env, Groq la levanta sola con: client = Groq()
# Si preferís asegurarte de que funcione ya mismo, pegala directo acá abajo:
client = Groq() 

def run_react_agent(pregunta: str, max_iterations: int = 5) -> str:
    # Simulamos el chat creando un historial que arranca con el PROMPT_REACT
    historial_mensajes = [
        {"role": "system", "content": PROMPT_REACT},
        {"role": "user", "content": pregunta}
    ]

    for i in range(max_iterations):
        # Llamada a la API de Groq con el freno de mano puesto
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=historial_mensajes,
            temperature=0.0,  # Para que sea súper preciso con el formato de las acciones
            stop=["PAUSA", "Observación:"]  # <-- LE AGREGAMOS ESTO
        )
        response_text = completion.choices[0].message.content.strip()

        print(f"\n--- Iteración {i+1} ---")
        print(f"Modelo pensó/respondió:\n{response_text}\n")

        # 1. Verificar si es la respuesta final
        response_match_final = re.search(r"Respuesta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        # 2. Si no es respuesta final, buscar la Acción
        match = re.search(r"Acción:\s*(\w+)(?::\s*([^\n]*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacion_de_accion = ""

            if action_name == "consultar_stock":
                observacion_de_accion = consultar_stock(action_arg)
            elif action_name == "consultar_precio_producto":
                observacion_de_accion = consultar_precio_producto(action_arg)
            elif action_name == "encontrar_producto_mas_costoso":
                observacion_de_accion = herramienta_encontrar_producto_mas_costoso()
            elif action_name == "calcular_valor_total_lista":
                observacion_de_accion = herramienta_calcular_valor_total_lista(action_arg)
            else:
                observacion_de_accion = f"Error: Acción '{action_name}' desconocida. Verifica la implementación."

            print(f"Ejecutó acción: {action_name} con argumento '{action_arg}'")
            print(f"Observación: {observacion_de_accion}\n")

            # Alimentamos el historial para mantener el contexto en el bucle
            historial_mensajes.append({"role": "assistant", "content": response_text})
            historial_mensajes.append({"role": "user", "content": f"Observación: {observacion_de_accion}"})

        else:
            return f"Error: No se pudo extraer una Acción o Respuesta del agente. Respuesta: {response_text}"

    return "Error: Se alcanzó el límite máximo de iteraciones sin una respuesta final."

In [19]:
import time

print("--- Iniciando la interacción con el Agente ReAct ---")
print("="*80 + "\n")

# Interacción 1: Consultar Stock
pregunta_1 = "¿Cuántos teclados tenemos en stock?"
print(f"**Interacción 1: {pregunta_1}**")
respuesta_1 = run_react_agent(pregunta_1)
print(f"\n**RESPUESTA FINAL DEL AGENTE 1:** {respuesta_1}\n")

print("="*80 + "\n")
time.sleep(10) # Pausa estratégica para engañar al límite de la API

# Interacción 2: Consultar Precio
pregunta_2 = "¿Cuál es el precio de un headset?"
print(f"**Interacción 2: {pregunta_2}**")
respuesta_2 = run_react_agent(pregunta_2)
print(f"\n**RESPUESTA FINAL DEL AGENTE 2:** {respuesta_2}\n")

print("="*80 + "\n")
time.sleep(10)

# Interacción 3: Item no encontrado en stock
pregunta_3 = "¿Tenemos sillas en stock?"
print(f"**Interacción 3: {pregunta_3}**")
respuesta_3 = run_react_agent(pregunta_3)
print(f"\n**RESPUESTA FINAL DEL AGENTE 3:** {respuesta_3}\n")

print("="*80 + "\n")
time.sleep(10)

# Interacción 4: Encontrar el Producto Más Costoso
pregunta_4 = f"¿Cuál es el producto más costoso?"
print(f"**Interacción 4: {pregunta_4}**")
respuesta_4 = run_react_agent(pregunta_4)
print(f"\n**RESPUESTA FINAL DEL AGENTE 4:** {respuesta_4}\n")

print("="*80 + "\n")
time.sleep(10)

# Interacción 5: Calcular el Valor Total de la Lista (NUEVA FUNCIONALIDAD)
pregunta_5 = "¿Cuál es el valor de un teclado, una impresora y una webcam?"
print(f"**Interacción 5: {pregunta_5}**")
respuesta_5 = run_react_agent(pregunta_5)
print(f"\n**RESPUESTA FINAL DEL AGENTE 5:** {respuesta_5}\n")

print("--- Fin de las Interacciones ---")

--- Iniciando la interacción con el Agente ReAct ---

**Interacción 1: ¿Cuántos teclados tenemos en stock?**

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Para saber cuántos teclados hay en stock, debo usar la acción `consultar_stock` con "teclado" como argumento.
Acción: consultar_stock: teclado
PAUSA

Ejecutó acción: consultar_stock con argumento 'teclado'
Observación: Tenemos 120 teclados en stock.


--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: Hay 120 teclados en stock.


**RESPUESTA FINAL DEL AGENTE 1:** Hay 120 teclados en stock.


**Interacción 2: ¿Cuál es el precio de un headset?**

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_precio_producto para saber el precio de un headset.
Acción: consultar_precio_producto: headset
PAUSA

Ejecutó acción: consultar_precio_producto con argumento 'headset'
Observación: El precio de un(a) headset es USD 180.00.


--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: El preci

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 15.603093939s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 15
}
]

In [20]:
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta final.

Usa "Pensamiento" para describir tu razonamiento sobre el paso actual.
Usa "Acción" para ejecutar herramientas, y luego retorna obligatoriamente la palabra "PAUSA".
La "Observación" será el resultado de la acción que fue ejecutada en Python.

Acciones disponibles:
- consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej: "consultar_stock: teclado").
- consultar_precio_producto: devuelve el precio unitario de un producto (ej: "consultar_precio_producto: mouse de gamer").
- encontrar_producto_mas_costoso: devuelve el nombre y el precio del producto más costoso del inventario (no requiere argumentos).
- calcular_valor_total_lista: calcula el valor total de una lista de artículos de compra. Recibe una cadena con artículos separados por comas (ej: "calcular_valor_total_lista: teclado, mouse de gamer, monitor").

Ejemplo 1:
Pregunta: ¿Cuántos monitores tenemos en stock?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA
Observación: Tenemos 75 monitores en stock.
Respuesta: Hay 75 monitores en stock.

Ejemplo 2:
Pregunta: ¿Cuál es el producto más costoso?
Pensamiento: Necesito usar la acción encontrar_producto_mas_costoso para descubrir qué producto tiene el mayor precio.
Acción: encontrar_producto_mas_costoso
PAUSA
Observación: El producto más costoso es el monitor con un precio de USD 999.90.
Respuesta: El producto más costoso es el monitor con un precio de USD 999.90.

Ejemplo 3:
Pregunta: ¿Cuánto cuesta un teclado y un mouse de gamer?
Pensamiento: El usuario quiere saber el valor total de varios artículos. Debo usar la acción calcular_valor_total_lista con los artículos "teclado, mouse de gamer".
Acción: calcular_valor_total_lista: teclado, mouse de gamer
PAUSA
Observación: El valor total de los artículos encontrados es USD 249.50.
Respuesta: El valor total del teclado y del mouse de gamer es USD 249.50.
""".strip()

In [24]:
# Interacción 1: Consultar Stock
pregunta_1 = "¿Cuántos teclados tenemos en stock?"
respuesta_1 = run_react_agent(pregunta_1)
print(f"**RESPUESTA FINAL 1:** {respuesta_1}")

⚠️ API saturada al iniciar. Esperando 30 segundos de enfriamiento...

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: El usuario quiere saber la cantidad de teclados en stock. Debo usar la acción consultar_stock con el argumento "teclado".
Acción: consultar_stock: teclado
PAUSA

Ejecutó acción: consultar_stock con argumento 'teclado'
Observación: Tenemos 120 teclados en stock.

🛑 Error de Cuota detectado en Iteración 2 (Intento 1/3).
Esperando 45 segundos para que Google libere la IP...
🛑 Error de Cuota detectado en Iteración 2 (Intento 2/3).
Esperando 45 segundos para que Google libere la IP...
🛑 Error de Cuota detectado en Iteración 2 (Intento 3/3).
Esperando 45 segundos para que Google libere la IP...


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 58.790309945s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 58
}
]

In [25]:
%pip install groq

Note: you may need to restart the kernel to use updated packages.


In [30]:
# Interacción 1: Consultar Stock
pregunta_1 = "¿Cuántos teclados tenemos en stock?"
respuesta_1 = run_react_agent(pregunta_1)
print(f"**RESPUESTA FINAL 1:** {respuesta_1}")


--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de teclados.
Acción: consultar_stock: teclado

Ejecutó acción: consultar_stock con argumento 'teclado'
Observación: Tenemos 120 teclados en stock.


--- Iteración 2 ---
Modelo pensó/respondió:
Respuesta: Hay 120 teclados en stock.

**RESPUESTA FINAL 1:** Hay 120 teclados en stock.


In [31]:
def iniciar_conversacion_con_agente():
    print("--- Agente de Inventario Interactivo ---")
    print("Realiza tu pregunta sobre el inventario, o digita 'salir' para cerrar la sesión.")
    print("-" * 50)

    while True:
        pregunta_usuario = input("\nUsted: ")

        if pregunta_usuario.lower().strip() == 'salir':
            # Corregido: "incluidos" va sin tilde en el mensaje de cierre
            print("\nAtención finalizada. ¡Hasta pronto!")
            print("=" * 50)
            break

        print("\nAgente: Procesando...")
        try:
            # Llama a tu función que ahora tiene el stop de Groq corregido
            respuesta_agente = run_react_agent(pregunta_usuario)
            print(f"\nAgente: {respuesta_agente}")
        except Exception as e:
            print(f"\nAgente: Ocurrió un error al procesar su pregunta: {e}")
            print("Por favor, intenta nuevamente, o digita 'salir'.")

In [32]:
iniciar_conversacion_con_agente()


--- Agente de Inventario Interactivo ---
Realiza tu pregunta sobre el inventario, o digita 'salir' para cerrar la sesión.
--------------------------------------------------

Agente: Procesando...

--- Iteración 1 ---
Modelo pensó/respondió:
Pensamiento: Para saber el precio de dos monitores, primero debo averiguar el precio unitario de un monitor y luego multiplicarlo por dos. Debo usar la acción consultar_precio_producto para obtener el precio de un monitor.

Acción: consultar_precio_producto: monitor

Ejecutó acción: consultar_precio_producto con argumento 'monitor'
Observación: El precio de un(a) monitor es USD 999.90.


--- Iteración 2 ---
Modelo pensó/respondió:
Pensamiento: Ahora que sé que el precio de un monitor es USD 999.90, puedo calcular el precio de dos monitores multiplicando el precio unitario por 2.

Acción: calcular_valor_total_lista: monitor, monitor

Ejecutó acción: calcular_valor_total_lista con argumento 'monitor, monitor'
Observación: El valor total de los items e